# Exercise 3

## Refactoring mcp_chatbot for A2A

The `mcp_chatbot.py` is already import-safe because asyncio.run(main()) is under:

```python
if __name__ == "__main__":
```

However, the current `MCP_ChatBot only exposes an interactive `chat()` loop. An A2A executor needs a method that:
1. Accepts one query (no interaction loop logic).
2. Returns one response as a string.
3. Does not call `input()` or `print()` for the actual response.
4. Correctly creates and closes the MCP connection.

In this exercise, you will code an `answer_query()` method that implements everything that was mentioned above.

### Objectives
1) **Check if user query is a threat, block if threat**

2) **Parse the workflow or fallback return result = `{"text": res, "html": html}`**

3) **Return `fallback` if no `workflow` was identified**

4) **Setup server parameters**

5) **Create the client session**
    - **Initialize async session**
    - **Get tools (Optional)**
    - **Single turn chat logic  return result = `{"text": res, "html": html}`**


<hr>
<h4 style="color:green; font-weight:bold;">TIPS:</h4>

Below is the  `MCP_ChatBot` class that exposes an `answer_query()` method

For this **exercise**, fill in the missing code inside the blocks: `### START YOUR CODE HERE ###` and `### END YOUR CODE HERE ###`

The instuctions are given as comments.

**Do not add or change any code that is outside these blocks**. You may add new cells to experiment

The `chat()` and `connect_to_server_and_run()` methods were intentionally included to guide you on what to write for this exercise

The solution is also provided in `mcp_chatbot_solution.py` which you can use if all else fails. But **please try your best first** before taking a peak into the solution

<hr>

##### Cell 1

In [ ]:
%%writefile mcp_chatbot.py

from utils import is_threat
from openai import OpenAI
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal, Optional, List
from datetime import datetime
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
import asyncio
import json
import sys

load_dotenv()

client = OpenAI(
    api_key=os.getenv("BEDROCK_KEY"),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1"
)

class WorkflowChoice(BaseModel):
    choice: Optional[Literal[
        "txt2sql_workflow",
        "rag_workflow",
        "waypoints_workflow",
        "nearest_labs_workflow",
        "analysis_workflow"        
    ]] = Field(
        default = None,
        description = """`txt2sql_workflow`: the user is asking for information on the location of laboratories and agencies as well as the services that they offer.
        `waypoints_workflow`: the user is asking about how to go to a particular agency from a certain location.
        `rag_workflow`: the user is asking for client steps, processes, requirements, or information on how to avail of a particular service in an agency
        `nearest_labs_workflow`: the user is asking for nearest laboratories and/or the services that they offer given a reference location
        `analysis_workflow`: the user is asking about hotspot and service area analysis questions (data is for provincial level analysis only)
        """
    )
    fallback: Optional[str] = Field(default=None, description="Fallback response if the user's query does not fall in any of the given choices")

system_prompt = """Your task is to choose the appropriate workflow depending on the user's intent.
`txt2sql_workflow`: the user is asking for information on the location of laboratories and agencies as well as the services that they offer.
`waypoints_workflow`: the user is asking about how to go to a particular agency from a certain location.
`rag_workflow`: the user is asking for client steps, processes, requirements, or information on how to avail of a particular service in an agency
`nearest_labs_workflow`: the user is asking for nearest laboratories and/or the services that they offer given a reference location
`analysis_workflow`: the user is asking about hotspot and service area analysis questions

For more context, these are some of the queries that can be processed by the workflows:
`What services does DOST-ITDI offer?`
`What do I need to prepare for pipe stiffness test for pvc in DOST-ITDI`
`How do I get to DOST-ASTI from SMDC Light Residences`
`10 nearest laboratories to SMDC Light Residences that offer coliform count`
`Which provinces are classified as potentially underserved`

If the user's intent is does not fall in any of the workflows, return a fallback reply highlighting allowed questions.
"""

class MCP_ChatBot:

    def __init__(self):
        self.session: ClientSession = None
        self.available_tools: List[dict] = []

    async def get_intent(self, user_query):
        response = client.responses.parse(
            model="openai.gpt-5.6-luna",
            input = [
                {
                    "role":"system",
                    "content": system_prompt
                },
                {
                    "role":"user",
                    "content": user_query
                }
            ],
            text_format = WorkflowChoice
        )
        return response.output_parsed


    async def answer_query(self, user_query: str):
        pass
        ### START YOUR CODE HERE ###

        #1) Check if user query is a threat, block if threat

        #2) Parse the workflow or fallback return result = {"text": res, "html", html}

        #3) Return fallback if no workflow was identified

        #4) Setup server parameters

        #5) Create the client session
            #a) Initialize async session
            #b) Get tools (Optional)
            #c) Single turn chat logic  return result = {"text": res, "html": html}

        ### END YOUR CODE HERE ###

    async def chat(self):
        if datetime.now().hour < 12:
            time = "morning"
        elif datetime.now().hour >= 12:
            time = "afternoon"
        else:
            time = "evening"
        print(f"\nOneLab Agent:\nGood {time}! How may I assist you?")

        while True:

            user_query = input("Enter your query: ")
            print(f"\nUser:\n{user_query}")

            if user_query.lower() == "quit":
                break

            _is_threat, errors = is_threat(user_query)

            if _is_threat:
                print("\nOneLab Agent:\nI'm sorry, but I couldn't process your request as it was potentially unsafe. If you think this was a mistake, please try rephrasing your prompt or providing more context so I can better understand your request.")
                continue

            response = await self.get_intent(user_query)
            workflow = response.choice
            fallback = response.fallback

            if workflow:
                response = await self.session.call_tool(workflow, arguments = {"user_query":user_query})
                #result, html = response.structuredContent["result"]
                if response.isError:
                    print(response.content[0].text)
                    return

                if response.structuredContent is not None:
                    result, html = response.structuredContent["result"]
                else:
                    # Fallback to TextContent
                    texts = [c.text for c in response.content]

                    result = texts[0] if len(texts) > 0 else None
                    html = texts[1] if len(texts) > 1 else None
                print(f"\nOneLab Agent:\n{result}")
                if html:
                    print("---\nOpen `output.html` for the visualization.")
            elif fallback:
                print(f"\nOneLab Agent:\n{fallback}")

    async def connect_to_server_and_run(self):
        # Create server parameters for stdio connection
        server_script = (Path(__file__).resolve().parent / "../onelab_server/onelab_server.py").resolve()
        server_params = StdioServerParameters(
            command=sys.executable,  # Executable
            args=[str(server_script)],  # Optional command line arguments
            env=None,  # Optional environment variables
        )
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                self.session = session
                # Initialize the connection
                await session.initialize()

                # List available tools
                response = await session.list_tools()

                tools = response.tools
                print("\nConnected to server with tools:", [tool.name for tool in tools])

                self.available_tools = [{
                    "type": "function",
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema
                } for tool in response.tools]

                await self.chat()

async def main():
    chatbot = MCP_ChatBot()
    await chatbot.connect_to_server_and_run()


if __name__ == "__main__":
    asyncio.run(main())


## Sanity check for mcp server path

Below is a `sanity check` to see if the path to the mcp server exists

##### Cell 2

In [ ]:
from pathlib import Path
import mcp_chatbot

server_path = (
    Path(mcp_chatbot.__file__).resolve().parent
    / "../onelab_server/onelab_server.py"
).resolve()

print(server_path)
print(server_path.exists())

## Check if your answer to the exercise works

##### Cell 3

In [ ]:
from mcp_chatbot import MCP_ChatBot

##### Cell 4

In [ ]:
agent = MCP_ChatBot()

prompt = "laboratories in quezon city"

result = await agent.answer_query(prompt)

##### Cell 5

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result["text"]))

## Wrap the OneLab MCP Chatbot Agent in A2A

##### Cell 6

In [ ]:
%%writefile onelab_agent.py
import os
import uvicorn
import uuid
import json

from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
    Artifact,
    Part,
    TextPart,
    TaskArtifactUpdateEvent
)
from a2a.utils import new_agent_text_message

#Use the mcp_chatbot_solution if all else fails
#from mcp_chatbot_solution import MCP_ChatBot
from mcp_chatbot import MCP_ChatBot


class OneLabAgentExecutor(AgentExecutor):
    def __init__(self) -> None:
        self.agent = MCP_ChatBot()

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        prompt = context.get_user_input()
        try:
            result = await self.agent.answer_query(prompt)
            text = result["text"]
            html = result["html"]

            response_dict = {
                "text": text,
                "html": html
            }

            response = json.dumps(response_dict)

            message = new_agent_text_message(response)
            await event_queue.enqueue_event(message)

        except Exception as exc:
            response = {
                "text": f"Unable to process the request: {exc}",
                "html": None
            }
            await event_queue.enqueue_event(
                new_agent_text_message(
                    json.dumps(response)
                )
            )
    async def cancel(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        pass

def main() -> None:
    print(f"Running OneLab Agent")
    PORT = 9999
    HOST = "localhost"

    skill = AgentSkill(
        id="onelab_assistance",
        name="OneLab Assistance",
        description="Provides information about OneLab laboratories, services, requirements for testing, directions, nearby laboratories, and service-area analysis (provincial level)",
        tags=["onelab", "laboratories", "laboratory services", "directions", "laboratory test requirements", "provincial level service-area analysis"],
        examples=[
            "What services does itdi and xprt offer",
            "Show me the laboratories in NCR",
            "What laboratories in laguna offer ash content test",
            "Laboratories in quezon city",
            "what is the process for mosquito larvicidal test in itdi",
            "administrative process in itdi for arsenic test for distilled water",
            "what do i need to prepare for pipe stiffness test for pvc in itdi ", 
            "how do i get to asti from smdc light residences",
            "directions to itdi from mall of asia",
            "10 nearest laboratories to SMDC light residences that offer coliform count",  
            "Where are onelab agencies concentrated",
            "Which provinces have the most onelab services",
            "which provinces have the broadest range of tests",
            "which provinces have no local onelab presence",
            "which provinces are classified as potentially underserved",
            "Can you give a numerical summary of the service area classification"
        ],
    )

    agent_card = AgentCard(
        name="OneLabAgent",
        description="OneLab laboratory and services information agent.",
        url=f"http://{HOST}:{PORT}/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text", "text/html"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[skill],
    )

    request_handler = DefaultRequestHandler(
        agent_executor=OneLabAgentExecutor(),
        task_store=InMemoryTaskStore(),
    )

    server = A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=request_handler,
    )

    uvicorn.run(server.build(), host=HOST, port=PORT)

    
if __name__ == '__main__':
    main()
        

Now to activate your configured A2A agent, you would need to run your agent server. You can run the agent server using `uv`:

- Open a terminal
- Navigate to the `Exercise-3` directory:
    - `cd Exercise-3`
    - `uv init`
- Activate the virtual environment:
    - `uv venv`
    - `source .venv/bin/activate`
- Install the additional dependencies:
    - `uv add "a2a-sdk<1.0" mcp openai requests uvicorn starlette sse-starlette rank-bm25 pandas ipython`
- Type `uv run a2a_policy_agent.py` to run the server and activate your A2A agent. 
- When running the agent for the first time, you will see that the virtual environment will first be created automatically and then the agent will run.

## Create the A2A client to communicate with the OneLab A2A Agent

##### Cell 7

In [ ]:
import httpx
from IPython.display import Markdown, display
from a2a.client import (
    Client,
    ClientConfig,
    ClientFactory,
    create_text_message_object,
)
from a2a.types import AgentCard, Artifact, Message, Task, TaskArtifactUpdateEvent
from a2a.utils.message import get_message_text
from pathlib import Path
import webbrowser
import json

##### Cell 8

In [ ]:
host = "localhost"
port = 9999

##### Cell 9

In [ ]:
def display_agent_card(agent_card: AgentCard) -> None:
    def esc(text: str) -> str:
        """Escapes pipe characters for Markdown table compatibility."""
        return str(text).replace("|", r"\|")

    # --- Part 1: Main Metadata Table ---
    md_parts = [
        "### Agent Card Details",
        "| Property | Value |",
        "| :--- | :--- |",
        f"| **Name** | {esc(agent_card.name)} |",
        f"| **Description** | {esc(agent_card.description)} |",
        f"| **Version** | `{esc(agent_card.version)}` |",
        f"| **URL** | [{esc(agent_card.url)}]({agent_card.url}) |",
        f"| **Protocol Version** | `{esc(agent_card.protocol_version)}` |",
    ]

    # --- Part 2: Skills Table ---
    if agent_card.skills:
        md_parts.extend(
            [
                "\n#### Skills",
                "| Name | Description | Examples |",
                "| :--- | :--- | :--- |",
            ]
        )
        for skill in agent_card.skills:
            examples_str = (
                "<br>".join(f"• {esc(ex)}" for ex in skill.examples)
                if skill.examples
                else "N/A"
            )
            md_parts.append(
                f"| **{esc(skill.name)}** | {esc(skill.description)} | {examples_str} |"
            )

    # Join all parts and display
    display(Markdown("\n".join(md_parts)))

### Display the agent card

##### Cell 10

In [ ]:
async with httpx.AsyncClient(timeout=100.0) as httpx_client:
    client: Client = await ClientFactory.connect(
        f"http://{host}:{port}",
        client_config=ClientConfig(
            httpx_client=httpx_client,
        ),
    )

    agent_card = await client.get_card()
    display_agent_card(agent_card)

##### Cell 11

In [ ]:
prompt = "what do i need to prepare for pipe stiffness test for pvc in itdi"

##### Cell 12

In [ ]:
async with httpx.AsyncClient(timeout=100.0) as httpx_client:

    client: Client = await ClientFactory.connect(
        f"http://{host}:{port}",
        client_config=ClientConfig(
            httpx_client=httpx_client,
        ),
    )

    agent_card = await client.get_card()
    #display_agent_card(agent_card)

    message = create_text_message_object(content=prompt)

    display(Markdown(f"**Sending prompt:** `{prompt}` to the agent..."))

    responses = client.send_message(message)

    text_content = ""
    html_saved = False

    async for response in responses:
        if isinstance(response, Message):
            # The agent replied directly with a final message
            print(f"[CLIENT LOG] Message ID: {response.message_id}")
            print(response)
            raw_content = get_message_text(response)

            dict_content = json.loads(raw_content)

            text_content = dict_content["text"]

            html = dict_content["html"]
                
            if html and not html_saved:
                with open("output.html", "w", encoding="utf-8") as f:
                    f.write(html)
                print("Successfully created output.html!")
                webbrowser.open(Path("output.html").resolve().as_uri())
                html_saved = True  # Prevent re-triggering on subsequent task updates
                

    display(Markdown("### Final Agent Response\n-----"))
    if text_content:
        display(Markdown(text_content))
    else:
        display(
            Markdown(
                """**No final text content received or task did not 
                complete successfully.**"""
            )
        )


<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 **Congratulations!**  

You have successfully created and interacted with a **chat agent** that utilizes **MCP** for usage of workflow tools and **A2A** for agent communication. Learning the fundamentals of how AI can move beyond simple text generation to actively orchestrating predetermined workflows as tools is a major milestone. By understanding the basics of MCP and A2A, you have unlocked the foundational mechanics required to build resilient, autonomous systems that can execute complex, multi-step operations.

</div>


